# Arbeitspaket (AP) 1: Management & Nutzung relationaler Daten

### Übungsprüfung 2 «FitLine» (Musterlösung)

### Persönliche Angaben (bitte ergänzen) und Prüfungsregeln

<table>
  <tr><td>Vorname:</td><td></td></tr>
  <tr><td>Nachname:</td><td></td></tr>
  <tr><td>Immatrikulationsnummer:</td><td></td></tr>
  <tr><td>Modul:</td><td>Data Science</td></tr>
  <tr><td>Prüfungsdatum / Raum / Zeit:</td><td>[TT.MM.2026] / Raum: [anpassen] / [hh:mm – hh:mm]</td></tr>
  <tr><td>Erlaubte Hilfsmittel:</td><td>Es gelten die Prüfungsregeln (PDF-Dokument auf EXAM-Moodle).</td></tr>
  <tr><td>Nicht erlaubt:</td><td>Es gelten die Prüfungsregeln (PDF-Dokument auf EXAM-Moodle).</td></tr>
  <tr><td>Technische Probleme:</td><td>Wenden Sie sich bei technischen Problemen umgehend an die Prüfungsaufsicht.</td></tr>
</table>

## Bewertungskriterien

**(max. erreichbare Punkte: 40)**

<table>
  <thead>
    <tr><th>Teil</th><th>Thema</th><th>Aufgaben</th><th>Punkte</th></tr>
  </thead>
  <tbody>
    <tr><td>A</td><td>Konzeptfragen: Datenmodellierung &amp; Design</td><td>A1–A3</td><td>12</td></tr>
    <tr><td>B</td><td>SQL-Fehleranalyse (fitness.db)</td><td>B1–B2</td><td>16</td></tr>
    <tr><td>C</td><td>SQL-Abfragen schreiben (fitness.db)</td><td>C1–C3</td><td>12</td></tr>
    <tr><td><b>Total</b></td><td></td><td><b>8 Aufgaben</b></td><td><b>40</b></td></tr>
    <tr><td>D</td><td><i>Zusatzübungen für Fortgeschrittene (freiwillig, nicht bewertet)</i></td><td>D1–D6</td><td>–</td></tr>
  </tbody>
</table>

### Bewertungsschema

Jede Aufgabe wird für sich und **als Ganzes** bewertet. Ihre Punktzahl steht in der Überschrift.

<table>
  <thead>
    <tr><th>Kategorie</th><th>Beschreibung</th><th>Bewertung</th><th>4 Pkt</th><th>8 Pkt</th></tr>
  </thead>
  <tbody>
    <tr><td>Korrekt</td><td>Vollständig und fachlich einwandfrei: Antwort bzw. Begründung korrekt und nachvollziehbar; bei SQL-Aufgaben liefert die SQL-Abfrage das richtige Ergebnis</td><td>100%</td><td>4</td><td>8</td></tr>
    <tr><td>Minimale Mängel</td><td>Kleine Fehler (z.B. falsche Sortierung, fehlende Rundung) oder Begründung ungenau bzw. unvollständig</td><td>75%</td><td>3</td><td>6</td></tr>
    <tr><td>Mittlere Mängel</td><td>Grössere Fehler (z.B. fehlende Spalten, falsche Filterung) oder Begründung fehlt bzw. ist fachlich falsch</td><td>50%</td><td>2</td><td>4</td></tr>
    <tr><td>Gravierende Mängel</td><td>Problem bzw. Ansatz erkennbar, aber Lösung oder Schlussfolgerung im Kern falsch (z.B. Problem richtig benannt, aber falsch behoben; im Code: ungeeigneter Lösungsweg wie falscher JOIN-Typ oder fehlende Tabelle)</td><td>25%</td><td>1</td><td>2</td></tr>
    <tr><td>Nicht lauffähig / nicht sinnvoll</td><td>Antwort fehlt, Code nicht lauffähig (z.B. Syntaxfehler) oder kein Bezug zur Aufgabe</td><td>0%</td><td>0</td><td>0</td></tr>
  </tbody>
</table>

Treffen mehrere Mängel zusammen, zählt der schwerste. Alternative, fachlich korrekte Lösungswege werden ebenfalls als korrekt gewertet.

## Hinweise zur Datenbank

**Datenbank:** PostgreSQL — Datenbank `postgres`, Benutzer `pgadmin`. Die Verbindung wird im Setup-Abschnitt aufgebaut und getestet.

**Backup:** `fitness.db` (SQLite) im gleichen Verzeichnis wie dieses Notebook — wird automatisch verwendet, falls keine PostgreSQL-Verbindung möglich ist.

**Tabellen:** `studios`, `trainers`, `members`, `courses`, `enrollments`, `payments`

**Datenbestand:** Kursanmeldungen 2024–2025, Zahlungen 2025. Datumswerte sind als Text im Format `YYYY-MM-DD` gespeichert.

Das ER-Diagramm und die Tabellenbeschreibungen finden Sie im Sheet **«ER-Diagram»** bzw. **«Description»** der beiliegenden Datei `fitness_data.xlsx`.

**Antworten auf Konzeptfragen** tragen Sie bitte direkt in die vorgesehenen Markdown-Zellen ein.

## Setup (wird nicht bewertet)

In [1]:
# Libraries
import os
import sqlite3
import pandas as pd
from sqlalchemy import create_engine, text

# Settings
import warnings
warnings.filterwarnings('ignore')

print('Working directory:', os.getcwd())

Working directory: /workspaces/DS_HS2026_LNW/DS_HS2026_LNW_I_Examples/AP01_Uebung2_FitLine


### Datenbankverbindung zu PostgreSQL

In [2]:
# Zugangsdaten
user     = 'pgadmin'
password = 'geheim'
port     = '5432'
database = 'postgres'

# Host ermitteln: im Devcontainer heisst der PostgreSQL-Service 'db',
# bei Docker-in-Docker ist er unter 'localhost' erreichbar.
host = None
for candidate in ('db', 'localhost'):
    try:
        probe = create_engine(f'postgresql://{user}:{password}@{candidate}:{port}/{database}',
                              connect_args={'connect_timeout': 3})
        with probe.connect() as connection:
            connection.execute(text('SELECT 1'))
        probe.dispose()
        host = candidate
        break
    except Exception:
        pass

# Connection URL für alle folgenden Abfragen
if host is not None:
    db_connection_url = f'postgresql://{user}:{password}@{host}:{port}/{database}'
else:
    # Kein PostgreSQL erreichbar (z.B. lokal ohne Codespace) -> SQLite-Backup verwenden
    db_connection_url = 'sqlite:///fitness.db'
    print('Hinweis: Keine PostgreSQL-Verbindung gefunden - verwende SQLite-Backup fitness.db')

# Test der Datenbankverbindung
engine = create_engine(db_connection_url)
with engine.connect() as connection:
    print('Datenbank:', engine.dialect.name, '|', host if host else 'fitness.db')
engine.dispose()

Datenbank: postgresql | localhost


### Daten nach PostgreSQL laden

In [3]:
# Tabellen aus der SQLite-Datei fitness.db nach PostgreSQL übertragen
tables = ['studios', 'trainers', 'members', 'courses', 'enrollments', 'payments']

if db_connection_url.startswith('postgresql'):
    sqlite_engine = create_engine('sqlite:///fitness.db')
    engine = create_engine(db_connection_url)

    for table in tables:
        df = pd.read_sql_query(f'SELECT * FROM {table}', con=sqlite_engine)
        df.to_sql(table, engine, if_exists='replace', index=False)
        print(f'{table:<13}{len(df):>5} Zeilen')

    sqlite_engine.dispose()
    engine.dispose()
else:
    print('SQLite-Backup aktiv - die Daten liegen bereits in fitness.db.')

studios          5 Zeilen
trainers        14 Zeilen
members        420 Zeilen
courses         26 Zeilen
enrollments   6200 Zeilen
payments      5020 Zeilen


> **Hinweis — Backup mit SQLite**
>
> Falls die PostgreSQL-Verbindung nicht funktioniert, verwendet das Setup automatisch die SQLite-Datenbank `fitness.db`.
> Tabellen und Inhalte sind identisch, und alle Abfragen dieser Übung laufen unverändert in beiden Datenbanken.

---

<div style="background-color:#2E4057; color:white; padding:16px 24px; border-radius:6px; font-size:18px; margin-top:8px;">
&#9654;&nbsp; <strong>Ab hier beginnt die bewertete Prüfung — 40 Punkte</strong><br>
<span style="font-size:15px; font-weight:normal;">Tragen Sie Ihre Antworten direkt in die vorgesehenen Zellen ein und führen Sie alle Code-Zellen aus.</span>
</div>

---
## Teil A – Konzeptfragen (12 Punkte)

*Antworten ohne Bezug zum Schema von `fitness.db` werden nicht gewertet.*

### A1 (4 Punkte)

Ein Kollege findet die Tabelle `enrollments` überflüssig: *«Wir könnten einfach eine Spalte `courseid` in der Tabelle `members` ergänzen. Dann sehen wir direkt, welchen Kurs ein Mitglied besucht.»*

Welche Art von Beziehung besteht zwischen `members` und `courses`? Begründen Sie, warum der Vorschlag nicht funktioniert, und nennen Sie mindestens eine Information, die dabei verloren ginge.

**Ihre Antwort:**

Zwischen `members` und `courses` besteht eine **n:m-Beziehung** (viele-zu-viele): Ein Mitglied besucht mehrere Kurse (z.B. Yoga am Montag und Spinning am Freitag), und ein Kurs hat viele Mitglieder. Eine einzelne Spalte `courseid` in `members` kann pro Mitglied aber nur **einen** Kurs speichern. Für weitere Kurse müsste man das Mitglied mehrfach anlegen (Redundanz, Duplikate) oder Spalten wie `courseid_1`, `courseid_2` … ergänzen (schlechtes Design, nicht beliebig erweiterbar).

Eine n:m-Beziehung wird in relationalen Datenbanken über eine **Zwischentabelle** (Verbindungstabelle) aufgelöst, hier `enrollments`, mit zwei Fremdschlüsseln (`memberid`, `courseid`).

**Verlorene Informationen:** Die Zwischentabelle speichert Eigenschaften **der Beziehung selbst**, nämlich das Datum der Lektion (`enroll_date`) und ob das Mitglied erschienen ist (`attended`). Diese gehören weder zum Mitglied noch zum Kurs allein und gingen verloren, damit auch z.B. die Teilnahmequote (C3).

### A2 (4 Punkte)

Das Management meldet: *«FitLine hat 420 Mitglieder.»* Eine Analyse zeigt jedoch, dass 4 E-Mail-Adressen in `members` **zweimal** vorkommen, jeweils mit unterschiedlicher `memberid` (z.B. `M0017` und `M0417`, gleiche Person).

Beschreiben Sie das Datenproblem, nennen Sie die passende Data-Quality-Eigenschaft aus den Vorlesungsfolien (z.B. accurate, consistent, complete, reliable) und schlagen Sie eine Massnahme vor, mit der die Datenbank solche Fälle künftig verhindert.

**Ihre Antwort:**

Es handelt sich um **Duplikate**: Dieselbe Person wurde zweimal als Mitglied erfasst (z.B. bei einer erneuten Online-Registrierung). Weil der Primärschlüssel `memberid` jeweils neu vergeben wurde, verhindert er das nicht. Er garantiert nur eindeutige IDs, nicht eindeutige Personen. Die Kennzahl «420 Mitglieder» ist deshalb **nicht accurate**: Tatsächlich sind es **416 Personen**. Auch Auswertungen pro Mitglied (Umsatz, Kursbesuche) werden auf zwei Datensätze verteilt und damit verfälscht.

**Massnahme:** Ein `UNIQUE`-Constraint auf `members.email`. Die Datenbank lehnt dann einen zweiten Eintrag mit derselben E-Mail-Adresse ab. Bestehende Duplikate müssen vorher zusammengeführt werden (vgl. D2).

### A3 (4 Punkte)

Im alten System von FitLine war der Monatsbeitrag als Text gespeichert, z.B. `'CHF 89.00'`, `'CHF 109.00'`, `'CHF 49.00'`.

Nennen Sie **zwei konkrete Probleme**, die bei Auswertungen mit dieser Speicherform entstehen, und den geeigneten Datentyp für Geldbeträge.

**Ihre Antwort:**

1. **Keine Berechnungen:** `SUM(monthly_fee)` oder `AVG(monthly_fee)` funktionieren nicht (Fehler oder falsches Ergebnis), weil die Datenbank `'CHF 89.00'` nicht als Zahl erkennt. Jede Auswertung bräuchte zuerst eine fehleranfällige Umwandlung (Präfix entfernen, `CAST`).
2. **Falsche Sortierung und falsche Vergleiche:** Text wird zeichenweise verglichen. `'CHF 109.00'` < `'CHF 49.00'` < `'CHF 89.00'`, weil `'1'` vor `'4'` und `'8'` kommt. `ORDER BY` oder `WHERE monthly_fee > 'CHF 80'` liefern dann falsche Ergebnisse.

**Geeigneter Datentyp:** `NUMERIC(10,2)` bzw. `DECIMAL`, also ein exakter Dezimaltyp mit zwei Nachkommastellen. Die Währung gehört, falls nötig, in eine eigene Spalte. (Gleitkommatypen wie `FLOAT` können bei Geldbeträgen Rundungsfehler verursachen.)

---
## Teil B – SQL-Fehleranalyse (16 Punkte)

*Jede Abfrage läuft fehlerfrei durch, liefert aber ein falsches oder irreführendes Geschäftsergebnis. Die tatsächlichen Ergebnisse aus `fitness.db` sind angegeben.*

### B1 (8 Punkte)

**Ziel:** Anzahl Kursanmeldungen pro Trainerin bzw. Trainer — für **alle** Trainer in der Datenbank.

```sql
SELECT t.trainerid, t.firstname, t.lastname, COUNT(*) AS enrollment_count
FROM trainers t
JOIN courses c     ON t.trainerid = c.trainerid
JOIN enrollments e ON c.courseid  = e.courseid
GROUP BY t.trainerid, t.firstname, t.lastname;
```

Die Tabelle `trainers` enthält **14 Trainer**. Tatsächliches Ergebnis der SQL-Abfrage: nur **13 Zeilen**, Anzahl Anmeldungen zwischen rund 100 und 600.

Die Studioleitung fragt: *«Zeig mir auch Trainer, die noch keine Anmeldungen haben.»*

**Aufgabe:** Schreiben Sie die korrigierte SQL-Abfrage, sodass **alle 14 Trainer** erscheinen und ein Trainer ohne Anmeldungen korrekt mit `0` ausgewiesen wird. Begründen Sie jede Änderung gegenüber der ursprünglichen SQL-Abfrage.

**Begründung der Änderungen:**

`INNER JOIN` gibt nur Zeilen zurück, für die in **beiden** Tabellen eine Übereinstimmung existiert. Trainer **T14 (Matteo Meier)** leitet noch keinen Kurs. Er findet keine Entsprechung in `courses` und damit auch nicht in `enrollments` und fällt aus dem Ergebnis.

Zwei Änderungen sind nötig:

1. Beide `JOIN`s müssen zu **`LEFT JOIN`** werden, ausgehend von `trainers` als linker Tabelle. So bleiben alle 14 Trainer erhalten.
2. **`COUNT(*)` muss zu `COUNT(e.enrollmentid)` werden.** `COUNT(*)` zählt Zeilen und würde die durch den `LEFT JOIN` erzeugte Zeile mit `NULL`-Werten als `1` mitzählen. T14 erschiene fälschlich mit `enrollment_count = 1`. `COUNT(e.enrollmentid)` ignoriert `NULL` und liefert korrekt `0`.

*Hinweis zur Bewertung: Ein `ORDER BY` ist nicht verlangt.*

**Korrigierte SQL-Abfrage:**

In [4]:
engine = create_engine(db_connection_url)

df_B1 = pd.read_sql_query("""
SELECT
    t.trainerid,
    t.firstname,
    t.lastname,
    COUNT(e.enrollmentid) AS enrollment_count
FROM trainers t
LEFT JOIN courses c     ON t.trainerid = c.trainerid
LEFT JOIN enrollments e ON c.courseid  = e.courseid
GROUP BY t.trainerid, t.firstname, t.lastname
ORDER BY enrollment_count DESC
""", con=engine)

engine.dispose()
df_B1

,trainerid,firstname,lastname,enrollment_count
0,T03,Noah,Schmid,604
1,T07,Mia,Schmid,575
2,T02,Matteo,Bianchi,574
3,T09,Lea,Meyer,566
4,T01,Alina,Weber,539
5,T13,Noah,Graf,539
6,T11,Robin,Meier,530
7,T08,Robin,Frei,528
8,T12,Jonas,Meyer,527
9,T04,Laura,Brunner,511


### B2 (8 Punkte)

**Ziel:** Anzahl **verschiedener Mitglieder** pro Kurskategorie, die **2025** mindestens eine Lektion dieser Kategorie gebucht haben, absteigend sortiert.

```sql
SELECT c.category, COUNT(e.memberid) AS member_count
FROM courses c
JOIN enrollments e ON c.courseid = e.courseid
WHERE e.enroll_date BETWEEN '2025-01-01' AND '2025-12-31'
GROUP BY c.category
ORDER BY member_count DESC;
```

Tatsächliches Ergebnis:

| category | member_count |
|---|---|
| Spinning | 845 |
| HIIT | 826 |
| Yoga | 810 |
| Pilates | 545 |
| Rückentraining | 99 |

FitLine hat insgesamt nur **420 Mitglieder**.

**Aufgabe:** Schreiben Sie die korrigierte SQL-Abfrage. Begründen Sie jede Änderung gegenüber der ursprünglichen SQL-Abfrage.

**Begründung der Änderungen:**

Die Tabelle `enrollments` enthält eine Zeile pro **Anmeldung**, nicht pro Mitglied. Wer 2025 zehnmal im Spinning war, hat zehn Zeilen und wird mit `COUNT(e.memberid)` zehnmal gezählt. Deshalb sind die Zahlen grösser als die Gesamtzahl der Mitglieder (845 > 420). Die Abfrage zählt also Anmeldungen, nicht Mitglieder.

**Änderung:** `COUNT(e.memberid)` → **`COUNT(DISTINCT e.memberid)`**. `DISTINCT` zählt jede `memberid` pro Kategorie nur **einmal**.

Ergebnis: Spinning 328, Yoga 318, HIIT 318, Pilates 268, Rückentraining 85 verschiedene Mitglieder. Alle Werte liegen jetzt unter 420.

**Korrigierte SQL-Abfrage:**

In [5]:
engine = create_engine(db_connection_url)

df_B2 = pd.read_sql_query("""
SELECT
    c.category,
    COUNT(DISTINCT e.memberid) AS member_count
FROM courses c
JOIN enrollments e ON c.courseid = e.courseid
WHERE e.enroll_date BETWEEN '2025-01-01' AND '2025-12-31'
GROUP BY c.category
ORDER BY member_count DESC
""", con=engine)

engine.dispose()
df_B2

,category,member_count
0,Spinning,328
1,HIIT,318
2,Yoga,318
3,Pilates,268
4,Rückentraining,85


---
## Teil C – SQL-Abfragen schreiben (12 Punkte)

Schreiben Sie SQL-Abfragen auf `fitness.db` und begründen Sie Ihre Vorgehensweise jeweils in 1–2 Sätzen. Worauf die Begründung eingehen soll, steht bei der Aufgabe.

### C1 (4 Punkte)

Ein Mitglied sucht einen **Yoga- oder Pilates-Kurs** mit **mindestens 15 Plätzen** (`max_participants`). Geben Sie `courseid`, `title`, `category` und `max_participants` aller passenden Kurse aus, sortiert nach `courseid`. Begründen Sie Ihre Filterbedingung.

**Begründung der Vorgehensweise:**

Alle Spalten liegen in `courses`, ein JOIN ist nicht nötig. `category IN ('Yoga', 'Pilates')` ist eine Kurzform für `category = 'Yoga' OR category = 'Pilates'`. Mit `AND max_participants >= 15` müssen **beide** Bedingungen erfüllt sein. `>=` statt `>`, weil «mindestens 15» die 15 einschliesst.

*Achtung: Ohne `IN` bzw. ohne Klammern wäre `category = 'Yoga' OR category = 'Pilates' AND max_participants >= 15` falsch, weil `AND` stärker bindet als `OR`. Dann kämen alle Yoga-Kurse unabhängig von der Grösse.*

**SQL-Abfrage:**

In [6]:
engine = create_engine(db_connection_url)

df_C1 = pd.read_sql_query("""
SELECT
    courseid,
    title,
    category,
    max_participants
FROM courses
WHERE category IN ('Yoga', 'Pilates')
  AND max_participants >= 15
ORDER BY courseid
""", con=engine)

engine.dispose()
df_C1

,courseid,title,category,max_participants
0,C001,Yoga Flow,Yoga,20
1,C002,Power Yoga,Yoga,15
2,C008,Pilates Core,Pilates,15
3,C016,Pilates Mat,Pilates,20
4,C019,Yoga Flow,Yoga,15
5,C020,Power Yoga,Yoga,15


### C2 (4 Punkte)

Wie hoch ist der durchschnittliche Stundenlohn (`hourly_rate`) der Trainer pro Studio? Geben Sie den Namen des Studios (`name`) und den Durchschnittslohn (`avg_rate`) aus, gerundet auf 2 Dezimalstellen mit `ROUND(CAST(... AS NUMERIC), 2)`. Sortieren Sie absteigend nach `avg_rate`. Begründen Sie, welche Tabellen Sie verbinden müssen und warum.

**Begründung der Vorgehensweise:**

Der Studioname liegt in `studios`, der Stundenlohn in `trainers`. Ein JOIN über `studioid` verbindet beide Tabellen. `GROUP BY s.name` gruppiert die Trainer nach Studio, und `AVG(t.hourly_rate)` berechnet den Durchschnitt. `ROUND(CAST(... AS NUMERIC), 2)` rundet auf Rappen. Das `CAST` ist nötig, damit die Abfrage auch unter PostgreSQL läuft.

**SQL-Abfrage:**

In [7]:
engine = create_engine(db_connection_url)

df_C2 = pd.read_sql_query("""
SELECT
    s.name,
    ROUND(CAST(AVG(t.hourly_rate) AS NUMERIC), 2) AS avg_rate
FROM studios s
JOIN trainers t ON s.studioid = t.studioid
GROUP BY s.name
ORDER BY avg_rate DESC
""", con=engine)

engine.dispose()
df_C2

,name,avg_rate
0,FitLine Winterthur,78.00
1,FitLine Zürich HB,73.33
2,FitLine Basel,72.33
3,FitLine Oerlikon,72.00
4,FitLine Luzern,67.50


### C3 (4 Punkte)

Welche Studios haben **mehr als 1'500 Kursanmeldungen** (2024 und 2025 zusammen)? Geben Sie `name` und die Anzahl Anmeldungen (`enrollment_count`) aus, absteigend sortiert nach `enrollment_count`. Begründen Sie, welche Tabellen Sie verbinden müssen und warum.

**Begründung der Vorgehensweise:**

`enrollments` ist nicht direkt mit `studios` verknüpft, der Weg führt über `courses`. Zwei JOINs verbinden `studios → courses → enrollments`. `GROUP BY s.name` fasst alle Anmeldungen pro Studio zusammen. `HAVING COUNT(e.enrollmentid) > 1500` filtert danach die Studios, weil die Bedingung ein **Aggregat** betrifft. In `WHERE` ginge das nicht.

**SQL-Abfrage:**

In [8]:
engine = create_engine(db_connection_url)

df_C3 = pd.read_sql_query("""
SELECT
    s.name,
    COUNT(e.enrollmentid) AS enrollment_count
FROM studios s
JOIN courses c     ON s.studioid = c.studioid
JOIN enrollments e ON c.courseid = e.courseid
GROUP BY s.name
HAVING COUNT(e.enrollmentid) > 1500
ORDER BY enrollment_count DESC
""", con=engine)

engine.dispose()
df_C3

,name,enrollment_count
0,FitLine Oerlikon,1676
1,FitLine Winterthur,1671
2,FitLine Zürich HB,1570


---
## Teil D – Zusatzübungen für Fortgeschrittene (freiwillig, nicht Teil der 40 Punkte)

*Etwas anspruchsvoller als die Prüfungsaufgaben. Gut zur Vertiefung, wenn Teil A–C sitzt.*

### D1 – Fehleranalyse: NULL in Berechnungen

**Ziel:** Tatsächlich eingenommener Umsatz 2025 pro Studio (nach Rabatt). Berücksichtigt werden **nur bezahlte** Zahlungen (`status = 'paid'`), gerundet auf 2 Dezimalstellen.

```sql
SELECT s.name, SUM(p.amount - p.discount) AS revenue
FROM studios s
JOIN members m  ON s.studioid = m.studioid
JOIN payments p ON m.memberid = p.memberid
WHERE p.status <> 'refunded'
GROUP BY s.name
ORDER BY revenue DESC;
```

Tatsächliches Ergebnis: FitLine Basel CHF 12'529, die Buchhaltung meldet aber rund **CHF 80'000**. Hinweis: `discount` ist `NULL`, wenn kein Rabatt gewährt wurde.

**Aufgabe:** Korrigieren Sie die Abfrage und begründen Sie jede Änderung.

**Begründung der Änderungen:**

1. **`COALESCE(p.discount, 0)`:** Jede Rechnung mit `NULL` ergibt `NULL` (`89.00 - NULL = NULL`), und `SUM` ignoriert `NULL`. Darum werden nur die Zahlungen **mit** Rabatt summiert. `COALESCE` ersetzt `NULL` durch 0.
2. **`WHERE p.status = 'paid'`** statt `<> 'refunded'`, weil sonst auch offene (`pending`) Zahlungen mitgezählt werden.
3. **Rundung** mit `ROUND(CAST(... AS NUMERIC), 2)`.

Ergebnis: FitLine Basel CHF 80'826.00.

**Korrigierte SQL-Abfrage:**

In [9]:
engine = create_engine(db_connection_url)

df_D1 = pd.read_sql_query("""
SELECT
    s.name,
    ROUND(CAST(SUM(p.amount - COALESCE(p.discount, 0)) AS NUMERIC), 2) AS revenue
FROM studios s
JOIN members m  ON s.studioid = m.studioid
JOIN payments p ON m.memberid = p.memberid
WHERE p.status = 'paid'
GROUP BY s.name
ORDER BY revenue DESC
""", con=engine)

engine.dispose()
df_D1

,name,revenue
0,FitLine Basel,80826.0
1,FitLine Oerlikon,76083.0
2,FitLine Winterthur,67947.0
3,FitLine Zürich HB,66381.0
4,FitLine Luzern,54578.0


### D2 – Subquery

Welche Trainer verdienen **mehr als den durchschnittlichen Stundenlohn** aller Trainer? Geben Sie `trainerid`, `firstname`, `lastname` und `hourly_rate` aus, absteigend sortiert. Der Durchschnitt darf **nicht** als feste Zahl in die Abfrage geschrieben werden.

**Begründung der Vorgehensweise:**

Die **Subquery** `(SELECT AVG(hourly_rate) FROM trainers)` liefert genau einen Wert (CHF 73.00), mit dem jede Zeile verglichen wird. `WHERE hourly_rate > AVG(hourly_rate)` direkt geht nicht, weil Aggregatfunktionen in `WHERE` nicht erlaubt sind.

**SQL-Abfrage:**

In [10]:
engine = create_engine(db_connection_url)

df_D2 = pd.read_sql_query("""
SELECT
    trainerid,
    firstname,
    lastname,
    hourly_rate
FROM trainers
WHERE hourly_rate > (SELECT AVG(hourly_rate) FROM trainers)
ORDER BY hourly_rate DESC
""", con=engine)

engine.dispose()
df_D2

,trainerid,firstname,lastname,hourly_rate
0,T09,Lea,Meyer,95.0
1,T06,Zoe,Schneider,90.0
2,T03,Noah,Schmid,85.0
3,T12,Jonas,Meyer,80.0
4,T13,Noah,Graf,77.0
5,T05,Noah,Vogel,75.0


### D3 – Teilnahmequote

Wie hoch ist die Teilnahmequote (Anteil `attended = 1`) pro Kurskategorie im Jahr 2025? Geben Sie `category`, Anzahl Anmeldungen (`enrollments`) und die Quote in Prozent (`attendance_pct`, 1 Dezimalstelle) aus, absteigend sortiert.

**Begründung der Vorgehensweise:**

Weil `attended` nur 0 und 1 enthält, ist `AVG(e.attended)` genau der Anteil der Einsen. Mal 100 ergibt Prozent. *Achtung: `SUM(attended) / COUNT(*)` wäre eine Ganzzahl-Division und ergäbe 0. Richtig ist `SUM(attended) * 100.0 / COUNT(*)`.*

**SQL-Abfrage:**

In [11]:
engine = create_engine(db_connection_url)

df_D3 = pd.read_sql_query("""
SELECT
    c.category,
    COUNT(e.enrollmentid) AS enrollments,
    ROUND(CAST(AVG(e.attended) * 100 AS NUMERIC), 1) AS attendance_pct
FROM courses c
JOIN enrollments e ON c.courseid = e.courseid
WHERE e.enroll_date BETWEEN '2025-01-01' AND '2025-12-31'
GROUP BY c.category
ORDER BY attendance_pct DESC
""", con=engine)

engine.dispose()
df_D3

,category,enrollments,attendance_pct
0,Rückentraining,99,96.0
1,Yoga,810,86.2
2,Pilates,545,83.9
3,Spinning,845,78.3
4,HIIT,826,72.6


### D4 – Altersgruppen mit CASE

Wie viele Mitglieder gibt es pro Altersgruppe (Alter 2025 = `2025 - birth_year`)? Gruppen: `unter 25`, `25-39`, `40-59`, `60+`.

**Begründung der Vorgehensweise:**

`CASE WHEN` prüft die Bedingungen von oben nach unten, und die erste zutreffende gilt. Die Nummern vor den Gruppennamen sorgen für die richtige Sortierung.

**SQL-Abfrage:**

In [12]:
engine = create_engine(db_connection_url)

df_D4 = pd.read_sql_query("""
SELECT
    CASE
        WHEN 2025 - birth_year < 25 THEN '1: unter 25'
        WHEN 2025 - birth_year < 40 THEN '2: 25-39'
        WHEN 2025 - birth_year < 60 THEN '3: 40-59'
        ELSE '4: 60+'
    END AS age_group,
    COUNT(*) AS member_count
FROM members
GROUP BY age_group
ORDER BY age_group
""", con=engine)

engine.dispose()
df_D4

,age_group,member_count
0,1: unter 25,73
1,2: 25-39,122
2,3: 40-59,112
3,4: 60+,113


### D5 – Duplikate finden

Finden Sie alle E-Mail-Adressen, die in `members` mehr als einmal vorkommen (vgl. A2). Geben Sie `email`, Anzahl Konten (`accounts`) sowie die kleinste und grösste `memberid` aus.

**Begründung der Vorgehensweise:**

`GROUP BY email` fasst Konten mit gleicher E-Mail zusammen, und `HAVING COUNT(*) > 1` behält nur die doppelten.

**SQL-Abfrage:**

In [13]:
engine = create_engine(db_connection_url)

df_D5 = pd.read_sql_query("""
SELECT
    email,
    COUNT(*) AS accounts,
    MIN(memberid) AS first_id,
    MAX(memberid) AS second_id
FROM members
GROUP BY email
HAVING COUNT(*) > 1
ORDER BY email
""", con=engine)

engine.dispose()
df_D5

,email,accounts,first_id,second_id
0,david.bianchi203@example.com,2,M0203,M0419
1,david.bianchi350@example.com,2,M0350,M0420
2,matteo.vogel17@example.com,2,M0017,M0417
3,selina.roth88@example.com,2,M0088,M0418


### D6 – Umsatz pro Monat

Berechnen Sie den bezahlten Umsatz (nach Rabatt, nur `status = 'paid'`) pro Monat 2025. Geben Sie `month` (Format `YYYY-MM`) und `revenue` aus, sortiert nach Monat.

**Begründung der Vorgehensweise:**

`SUBSTR(payment_date, 1, 7)` schneidet aus `'2025-03-04'` den Monat `'2025-03'` heraus. Danach wird nach diesem Ausdruck gruppiert. Für den Rabatt braucht es wie in D1 `COALESCE`.

**SQL-Abfrage:**

In [14]:
engine = create_engine(db_connection_url)

df_D6 = pd.read_sql_query("""
SELECT
    SUBSTR(payment_date, 1, 7) AS month,
    ROUND(CAST(SUM(amount - COALESCE(discount, 0)) AS NUMERIC), 2) AS revenue
FROM payments
WHERE status = 'paid'
GROUP BY SUBSTR(payment_date, 1, 7)
ORDER BY month
""", con=engine)

engine.dispose()
df_D6

,month,revenue
0,2025-01,29195.0
1,2025-02,29204.0
2,2025-03,28540.0
3,2025-04,28653.0
4,2025-05,29253.0
5,2025-06,28552.0
6,2025-07,28402.0
7,2025-08,28119.0
8,2025-09,29323.0
9,2025-10,29016.0


---
### Jupyter notebook --footer info-- (please always provide this at the end of each notebook)

In [15]:
import os, platform, socket
from platform import python_version
from datetime import datetime

print('-----------------------------------')
print(os.name.upper())
print(platform.system(), '|', platform.release())
print('Datetime:', datetime.now().strftime('%Y-%m-%d %H:%M:%S'))
print('Python Version:', python_version())
print('IP Address:', socket.gethostbyname(socket.gethostname()))
print('-----------------------------------')

-----------------------------------
POSIX
Linux | 6.8.0-1064-azure
Datetime: 2026-11-02 10:45:11
Python Version: 3.12.11
IP Address: 127.0.0.1
-----------------------------------
